# StreamMind: Simulador de Viewers con Inteligencia Artificial para Streams en Vivo

**Curso:** Introduccion a la Inteligencia Artificial — EAFIT 2026  
**Fase:** Anteproyecto  
**Fecha de entrega:** Mayo 10, 2026
**integrantes:** Sara Ruiz - David Hernandez - Thomas Zambrano 
---

## 1. Problema e Idea

### Problema

Los streams en vivo (Twitch, YouTube Live, etc.) dependen en gran medida de la interaccion del chat para generar una experiencia dinamica. Cuando un streamer cuenta con pocos seguidores o transmite en horarios de baja audiencia, el chat queda vacio, lo que deteriora la percepcion del contenido tanto para el streamer como para los espectadores ocasionales que llegan al directo.

### Idea

Construir un sistema de inteligencia artificial que escuche la voz del streamer en tiempo real, comprenda el contexto de lo que ocurre en el stream, y genere automaticamente comentarios de chat que simulen la participacion de tres tipos distintos de viewers. Estos comentarios se sincronizan con el flujo del directo mediante un delay calculado, de modo que aparezcan en el chat en el momento correcto y con coherencia narrativa respecto a lo que el streamer acaba de decir.

El sistema utiliza un unico modelo de lenguaje grande (LLM) que, a traves de un prompt estructurado, adopta simultaneamente tres personalidades distintas y devuelve las tres respuestas en una sola inferencia. Esto lo hace significativamente mas rapido que usar tres modelos o tres llamadas separadas.

### Por que es interesante o util

La simulacion de engagement en streams es un problema real con aplicaciones en creacion de contenido, entretenimiento y marketing digital. Desde el punto de vista tecnico, el proyecto integra reconocimiento de voz, recuperacion aumentada por generacion (RAG), ingenieria de prompts avanzada y evaluacion de agentes de lenguaje, cubriendo los temas centrales del curso de forma cohesionada.

### Tipo de tarea

Generacion de lenguaje natural (NLP) con evaluacion basada en juez automatico (LLM-as-judge). El componente de evaluacion implica una tarea de clasificacion binaria: determinar si un comentario fue generado por el sistema o por un humano real.

## 2. Enfoque Propuesto

### Arquitectura del sistema

El sistema se compone de cuatro etapas principales que se ejecutan de forma secuencial en tiempo real:

**Etapa 1 — Transcripcion de voz (STT):** Se utiliza Whisper de OpenAI, un modelo de reconocimiento de voz preentrenado que corre localmente y de forma gratuita. Cada pocos segundos, el audio del microfono del streamer se transcribe a texto.

**Etapa 2 — Memoria contextual del stream (RAG):** La transcripcion acumulada del stream se almacena en un vector store usando FAISS y embeddings de `sentence-transformers`. Antes de generar cada respuesta, el sistema recupera los fragmentos mas relevantes del historial del stream para darle contexto al LLM. Esto evita que los comentarios sean genericos y hace que los viewers "recuerden" lo que se ha dicho.

**Etapa 3 — Generacion multi-persona con un solo LLM:** Se realiza una unica llamada al LLM (NVIDIA NIM con Llama 3.1, gratuito) con un prompt estructurado que le indica al modelo que adopte tres personalidades distintas y devuelva las tres respuestas en formato JSON:

- **HypeBot:** viewer entusiasta, usa jerga de gaming y reacciona de forma exagerada a los momentos clave.
- **CritiBot:** viewer analitico que hace preguntas inteligentes y comentarios reflexivos sobre lo que ocurre.
- **LurkerBot:** viewer silencioso que interviene poco pero con humor seco o referencias de memes.

Las tres respuestas se muestran en pantalla con delays artificiales entre si para simular que llegan de forma independiente.

**Etapa 4 — Interfaz grafica en vivo:** Todo se presenta en una interfaz construida con Gradio, donde se visualiza la transcripcion del streamer en tiempo real y el chat simulado con los tres viewers.

### Por que este enfoque tiene sentido

Usar un solo LLM con un prompt multi-persona es coherente con los principios de ingenieria de prompts vistos en clase: con el disenio correcto del system prompt, un modelo puede mantener voces distintas y coherentes dentro de una misma inferencia. Esto reduce la latencia a una tercera parte comparado con tres llamadas separadas, lo cual es critico en un sistema en tiempo real. El RAG garantiza que los comentarios sean contextualmente relevantes y no genericos, que es el principal diferenciador respecto a soluciones triviales.

## 3. Datos

### Fuente del dataset

Los datos se recolectaran directamente desde Twitch mediante su protocolo publico IRC, que permite leer el chat de cualquier canal publico sin necesidad de credenciales ni API key. Esto garantiza que los comentarios de referencia sean completamente reales, frescos y pertenezcan exactamente a la categoria de stream que el sistema va a simular.

La recoleccion se hara con la libreria `socket` de Python, conectandose al servidor IRC de Twitch (`irc.chat.twitch.tv`) en modo anonimo (usuario `justinfan` + numero aleatorio), lo cual es un mecanismo oficial y documentado por Twitch para lectores anonimos.

Como fuente complementaria verificada en Kaggle se puede usar el dataset **"Twitch Chat Messages"**, buscando directamente en https://www.kaggle.com/search?q=twitch+chat en caso de necesitar un conjunto pre-recolectado.

### Script de recoleccion (referencia)

El proceso de recoleccion se implementara en el notebook de implementacion. A modo ilustrativo, la conexion anonima funciona asi:

```python
import socket

server = "irc.chat.twitch.tv"
port   = 6667
nick   = "justinfan12345"   # usuario anonimo oficial de Twitch
canal  = "#xqc"             # canal publico a escuchar

sock = socket.socket()
sock.connect((server, port))
sock.send(f"NICK {nick}\r\n".encode())
sock.send(f"JOIN {canal}\r\n".encode())

# Leer mensajes en tiempo real
while True:
    respuesta = sock.recv(2048).decode("utf-8", errors="ignore")
    if "PRIVMSG" in respuesta:
        usuario = respuesta.split("!")[0][1:]
        mensaje = respuesta.split("PRIVMSG")[1].split(":", 1)[1].strip()
        print(f"{usuario}: {mensaje}")
```

### Que contiene el dataset recolectado

Cada mensaje capturado tendra los siguientes campos:

- `username`: nombre del usuario que escribio el mensaje
- `message`: el texto del comentario en el chat
- `timestamp`: momento exacto en que fue capturado
- `channel`: canal de Twitch del que proviene
- `stream_category`: categoria del stream (gaming, IRL, just chatting, etc.), obtenida via la API publica de Twitch

### Variable objetivo

Para el componente de evaluacion, la variable objetivo es binaria: `es_humano` (1 = comentario real capturado de Twitch, 0 = comentario generado por el sistema). Esta etiqueta permite evaluar si un juez automatico o humano puede distinguir los comentarios generados de los reales.

### Tamano aproximado

Se recolectaran entre 5.000 y 10.000 mensajes reales en sesiones de captura de 30 a 60 minutos sobre canales de distintas categorias, balanceando la muestra para tener representacion equitativa. Para la evaluacion final se usara un subconjunto de 50 comentarios reales emparejados con 50 comentarios generados por el sistema ante el mismo contexto de stream.

## 4. Exploracion Inicial de los Datos (EDA)

A continuacion se realiza una exploracion inicial sobre una muestra sintetica representativa del tipo de datos que se usaran. Esta muestra fue construida para reflejar las distribuciones tipicas de un dataset de chat de Twitch (longitud de mensajes corta, lenguaje informal, alta frecuencia de mensajes breves).

In [2]:
%pip install matplotlib pandas numpy -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

np.random.seed(42)

# Muestra sintetica representativa de un chat de Twitch
n = 500
categorias = ["gaming", "just chatting", "IRL", "music", "sports"]
longitudes = np.random.exponential(scale=18, size=n).astype(int).clip(1, 120).tolist()

mensajes_ejemplo = {
    "gaming":        ["let's gooo", "GG", "skill issue", "no way bro", "clutch", "KEKW", "nice play", "oof"],
    "just chatting": ["lol same", "fr fr", "true", "based", "bruh", "omg", "this guy", "real"],
    "IRL":           ["where are you?", "nice view", "haha", "what city?", "go bro", "lets go", "wow"],
    "music":         ["fire track", "W music", "banger", "lowkey slaps", "who is this?", "love this"],
    "sports":        ["lets go", "nooo", "what a goal", "ref is blind", "insane", "W", "gg"],
}

filas = []
for i in range(n):
    cat = np.random.choice(categorias)
    msg = np.random.choice(mensajes_ejemplo[cat])
    long = longitudes[i]
    msg_final = (msg * ((long // len(msg)) + 1))[:long]
    filas.append({
        "message":          msg_final,
        "stream_category":  cat,
        "char_length":      len(msg_final),
        "word_count":       len(msg_final.split()),
        "viewer_count":     int(np.random.lognormal(mean=6, sigma=1.5)),
    })

df = pd.DataFrame(filas)
print(f"Total de mensajes en la muestra: {len(df)}")
print(f"\nEstadisticas de longitud de mensajes (caracteres):")
print(df["char_length"].describe().round(2))
print(f"\nDistribucion por categoria:")
print(df["stream_category"].value_counts())


TypeError: The 'out' kwarg is necessary when using the string multiply ufunc directly. Use numpy.strings.multiply to multiply strings without specifying 'out'.

### Grafico 1: Distribucion de la longitud de los mensajes en el chat

Una de las observaciones mas importantes es que la gran mayoria de los mensajes de chat de Twitch son muy cortos (menos de 20 caracteres). Esto es un rasgo caracteristico del formato: los viewers escriben rapido, en bloques de pocas palabras. El sistema generador debera respetar esta distribucion para que los comentarios artificiales sean indistinguibles de los reales.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.hist(df["char_length"], bins=30, color="#4C72B0", edgecolor="white", linewidth=0.6)
ax.set_xlabel("Longitud del mensaje (caracteres)", fontsize=12)
ax.set_ylabel("Numero de mensajes", fontsize=12)
ax.set_title("Distribucion de longitud de mensajes en chat de Twitch (muestra sintetica)", fontsize=13)
ax.axvline(df["char_length"].median(), color="#DD4444", linestyle="--", linewidth=1.5,
           label=f"Mediana: {df['char_length'].median():.0f} caracteres")
ax.legend(fontsize=11)
ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.show()

print(f"Mediana de longitud: {df['char_length'].median():.0f} caracteres")
print(f"Porcentaje de mensajes con menos de 20 caracteres: {(df['char_length'] < 20).mean()*100:.1f}%")

### Grafico 2: Longitud promedio de mensajes por categoria de stream

Distintas categorias de stream generan patrones de chat distintos. Los streams de "just chatting" tienden a tener mensajes mas largos porque los viewers opinan sobre conversaciones. Los streams de gaming tienen mensajes mas cortos y reactivos. El sistema debera adaptarse a la categoria del stream para generar comentarios con la longitud y tono apropiados.

In [ ]:
avg_by_cat = df.groupby("stream_category")["char_length"].mean().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 4))

bars = ax.barh(avg_by_cat.index, avg_by_cat.values, color="#55A868", edgecolor="white", linewidth=0.6)

for bar, val in zip(bars, avg_by_cat.values):
    ax.text(val + 0.3, bar.get_y() + bar.get_height() / 2,
            f"{val:.1f}", va="center", fontsize=11)

ax.set_xlabel("Longitud promedio del mensaje (caracteres)", fontsize=12)
ax.set_title("Longitud promedio de mensajes por categoria de stream", fontsize=13)
ax.set_xlim(0, avg_by_cat.max() + 8)

plt.tight_layout()
plt.show()

### Observaciones del EDA

- La distribucion de longitud de mensajes es fuertemente asimetrica hacia la derecha: la mayoria de los mensajes reales tienen menos de 20 caracteres. Esto indica que el sistema no debe generar comentarios largos si quiere imitar el estilo natural del chat.
- Existe variacion en la longitud promedio segun la categoria del stream, lo que sugiere que el prompt del LLM debera incluir la categoria como contexto para calibrar el estilo de respuesta.
- No se observa desbalance significativo entre categorias en la muestra, aunque en el dataset real los streams de gaming son desproporcionadamente mas frecuentes. Esto debera tenerse en cuenta al construir el conjunto de evaluacion.

## 5. Pregunta y Objetivo

### Pregunta de investigacion

Dado el contexto de lo que dice un streamer en tiempo real, puede un sistema basado en un unico LLM con un prompt multi-persona generar comentarios de chat que sean indistinguibles de los comentarios escritos por viewers humanos reales?

### Objetivo

Construir e implementar el sistema StreamMind, que integra transcripcion de voz en tiempo real, recuperacion de contexto mediante RAG, y generacion de comentarios multi-persona con un solo LLM, y demostrar mediante una evaluacion cuantitativa que los comentarios generados superan un umbral de humanness del 60%, medido con LLM-as-judge sobre un conjunto de referencia de comentarios reales de Twitch.

## 6. Evaluacion

### Metrica principal: Humanness Score

Se construira un conjunto de evaluacion con 100 comentarios en total: 50 comentarios reales extraidos del dataset de Twitch (seleccionados con el mismo contexto de stream que el sistema recibira) y 50 comentarios generados por StreamMind ante los mismos fragmentos de audio.

El **humanness score** se define como:

```
humanness_score = (numero de comentarios generados clasificados como humanos) / 50
```

Un LLM-as-judge (configurado segun los principios de la Lecture 13) recibira cada comentario junto con el contexto del stream y debera indicar si el comentario parece escrito por un humano real o por una IA. Se usara una rubrica explicita con criterios como: naturalidad del lenguaje, coherencia con el contexto, longitud apropiada, uso de jerga propia del medio.

### Metrica secundaria: precision del clasificador humano

Como validacion adicional, se le presentara a un grupo de al menos 5 personas un test ciego con una mezcla de comentarios reales y generados, y se medira la tasa de error de clasificacion. Si los humanos se equivocan mas del 30% de las veces, el sistema se considera exitoso.

### Justificacion

Estas metricas son las mas apropiadas para un sistema de generacion de lenguaje natural donde no existe una respuesta correcta unica. La evaluacion basada en juez automatico permite escalar la evaluacion sin depender exclusivamente de anotadores humanos, y el test con personas reales da validez externa a los resultados. Ambos enfoques fueron cubiertos en la Lecture 13 del curso.

## 7. Referencias

1. Radford, A., Kim, J. W., Xu, T., Brockman, G., McLeavey, C., & Sutskever, I. (2022). *Robust Speech Recognition via Large-Scale Weak Supervision*. OpenAI. https://cdn.openai.com/papers/whisper.pdf

2. Lewis, P., Perez, E., Piktus, A., et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*. NeurIPS 2020. https://arxiv.org/abs/2005.11401

3. Zheng, L., Chiang, W. L., Sheng, Y., et al. (2023). *Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena*. NeurIPS 2023. https://arxiv.org/abs/2306.05685

4. NVIDIA NIM Documentation — Llama 3.1 API (gratuita con cuenta NVIDIA). https://build.nvidia.com/explore/discover

5. LangSmith & agentevals Documentation — LangChain. https://docs.smith.langchain.com

---

*Nota: este anteproyecto fue elaborado como parte del proyecto final del curso Introduccion a la Inteligencia Artificial, EAFIT 2026.*